# Supply Chain Fraud Detection — Data Generator

Generates synthetic seller, warehouse, order, and logistics data with **injected fraud rings**, then loads it into Snowflake.

**What this notebook does:**
1. Creates 500 sellers, 30 warehouses, 10,000 orders, 2,000 logistics records
2. Injects 10 fraud rings (5 sellers each sharing a bank account)
3. Uploads all data to Snowflake tables
4. Creates graph-ready tables for Neo4j Graph Analytics
5. Builds shared bank account edge table for WCC analysis

In [ ]:
!pip install pandas numpy faker snowflake-connector-python

## Step 1: Generate Synthetic Supply Chain Data

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker

fake = Faker('en_IN')
np.random.seed(42)

N_SELLERS    = 500
N_WAREHOUSES = 30
N_ORDERS     = 10000

cities    = ['Mumbai','Delhi','Bangalore','Hyderabad','Chennai',
             'Kolkata','Pune','Jaipur','Lucknow','Surat']
platforms = ['Amazon','Flipkart']
couriers  = ['BlueDart','Delhivery','Ekart','XpressBees','DTDC']

# --- Sellers ---
sellers = pd.DataFrame({
    'NODEID':       range(1, N_SELLERS+1),
    'SELLER_NAME':  [fake.company() for _ in range(N_SELLERS)],
    'PLATFORM':     np.random.choice(platforms, N_SELLERS),
    'CITY':         np.random.choice(cities, N_SELLERS),
    'GST_NUMBER':   [f"27{fake.bothify('??########Z#')}" for _ in range(N_SELLERS)],
    'BANK_ACCOUNT': [fake.bban() for _ in range(N_SELLERS)],
    'RETURN_RATE':  np.round(np.random.beta(2, 8, N_SELLERS), 4),
    'REG_DAYS_AGO': np.random.randint(30, 2000, N_SELLERS),
    'FRAUD_FLAG':   np.random.choice([0,1], N_SELLERS, p=[0.92, 0.08])
})

# Inject 10 fraud rings (5 sellers each sharing a bank account)
for i in range(10):
    shared_bank = fake.bban()
    idx = np.random.choice(sellers.index, 5, replace=False)
    sellers.loc[idx, 'BANK_ACCOUNT'] = shared_bank
    sellers.loc[idx, 'FRAUD_FLAG']   = 1

# --- Warehouses ---
warehouses = pd.DataFrame({
    'NODEID':         range(5001, 5001+N_WAREHOUSES),
    'WAREHOUSE_NAME': [f"WH-{c[:3].upper()}-{i}" for i,c
                       in enumerate(np.random.choice(cities, N_WAREHOUSES))],
    'CITY':           np.random.choice(cities, N_WAREHOUSES),
    'PIN_CODE':       [fake.postcode() for _ in range(N_WAREHOUSES)]
})

# --- Orders ---
orders = pd.DataFrame({
    'SOURCENODEID':    np.random.choice(sellers['NODEID'], N_ORDERS),
    'TARGETNODEID':    np.random.choice(warehouses['NODEID'], N_ORDERS),
    'ORDER_ID':        [f"ORD{i:07d}" for i in range(N_ORDERS)],
    'ORDER_VALUE':     np.round(np.random.lognormal(8, 1.2, N_ORDERS), 2),
    'DELIVERY_DELAY':  np.random.choice([0,1,2,3,5,7,10], N_ORDERS, p=[0.4,0.2,0.15,0.1,0.08,0.05,0.02]),
    'RETURN_CLAIMED':  np.random.choice([0,1], N_ORDERS, p=[0.85, 0.15]),
    'ROUTE_DEVIATION': np.random.choice([0,1], N_ORDERS, p=[0.9, 0.1])
})

# --- Logistics ---
logistics = pd.DataFrame({
    'SOURCENODEID': np.random.choice(warehouses['NODEID'], 2000),
    'TARGETNODEID': np.random.choice(warehouses['NODEID'], 2000),
    'COURIER':      np.random.choice(couriers, 2000),
    'TRANSIT_DAYS':  np.random.randint(1, 15, 2000),
    'LOSS_COUNT':    np.random.choice([0,0,0,0,1,1,2,3], 2000)
})

print(f"Sellers: {len(sellers)}, Warehouses: {len(warehouses)}")
print(f"Orders: {len(orders)}, Logistics: {len(logistics)}")
print(f"Fraud rings injected: 10 (50 sellers total)")

## Step 2: Connect to Snowflake

Replace the placeholders below with your Snowflake credentials.

In [ ]:
import snowflake.connector

conn = snowflake.connector.connect(
    account   = '<YOUR_ACCOUNT>',
    user      = '<YOUR_USER>',
    password  = '<YOUR_PASSWORD>',
    warehouse = 'SUPPLY_WH',
    database  = 'SUPPLY_CHAIN_DB',
    schema    = 'PUBLIC',
    role      = 'ACCOUNTADMIN'
)
cur = conn.cursor()
print('Connected to Snowflake!')

## Step 3: Load Data into Snowflake

In [ ]:
from snowflake.connector.pandas_tools import write_pandas

tables = {
    'SELLERS':    sellers,
    'WAREHOUSES': warehouses,
    'ORDERS':     orders,
    'LOGISTICS':  logistics
}

for table_name, df in tables.items():
    success, nchunks, nrows, _ = write_pandas(
        conn, df, table_name,
        database  = 'SUPPLY_CHAIN_DB',
        schema    = 'PUBLIC',
        overwrite = False
    )
    print(f"{table_name}: {nrows} rows | success={success}")

## Step 4: Build Shared Bank Account Edges

This creates edges between sellers who share the same bank account — used by WCC to detect shell entities.

In [ ]:
for table in ['SELLERS','WAREHOUSES','ORDERS','LOGISTICS']:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]} rows")

cur.execute("""
    CREATE OR REPLACE TABLE SHARED_BANK_EDGES AS
    SELECT
        s1.nodeId       AS sourceNodeId,
        s2.nodeId       AS targetNodeId,
        s1.bank_account AS shared_bank
    FROM SELLERS s1
    JOIN SELLERS s2
        ON  s1.bank_account = s2.bank_account
        AND s1.nodeId < s2.nodeId
""")
conn.commit()

cur.execute("SELECT COUNT(*) FROM SHARED_BANK_EDGES")
print(f"Suspicious bank-sharing pairs: {cur.fetchone()[0]}")

## Step 5: Create Graph-Ready Tables

These tables are formatted for Neo4j Graph Analytics Native App (node tables and relationship tables).

In [ ]:
cur.execute("""
    CREATE OR REPLACE TABLE SELLERS_GRAPH AS
    SELECT nodeId, return_rate, reg_days_ago, fraud_flag
    FROM SELLERS
""")

cur.execute("""
    CREATE OR REPLACE TABLE WAREHOUSES_GRAPH AS
    SELECT nodeId FROM WAREHOUSES
""")

cur.execute("""
    CREATE OR REPLACE TABLE ORDERS_GRAPH AS
    SELECT sourceNodeId, targetNodeId, order_value,
           delivery_delay, return_claimed, route_deviation
    FROM ORDERS
""")

cur.execute("""
    CREATE OR REPLACE TABLE SHARED_BANK_EDGES_GRAPH AS
    SELECT sourceNodeId, targetNodeId
    FROM SHARED_BANK_EDGES
""")

cur.execute("""
    GRANT SELECT, INSERT ON ALL TABLES IN SCHEMA SUPPLY_CHAIN_DB.PUBLIC
    TO DATABASE ROLE SUPPLY_CHAIN_DB.gds_db_role
""")

conn.commit()
print('Graph tables created!')
print('Permissions granted!')

## Step 6: Verify All Tables

In [ ]:
cur.execute("SHOW TABLES IN SUPPLY_CHAIN_DB.PUBLIC")
tables = cur.fetchall()
print(f"Total tables: {len(tables)}")
for t in tables:
    print(f"   {t[1]}")

In [ ]:
for table in ['SELLERS', 'WAREHOUSES', 'ORDERS', 'LOGISTICS', 'SHARED_BANK_EDGES']:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = cur.fetchone()[0]
    print(f"{table}: {count} rows in Snowflake")

conn.close()
print('Done! Connection closed.')